# Machine Learning 2025W — Exercise 1  
### Dataset Description and Exploration (heart disease)

**Group Members:**  
- Full Name 1  
- Full Name 2  
- Full Name 3  

---


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
import ssl
import certifi
import itertools
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.metrics import classification_report
from sklearn.tree import plot_tree


ssl_context = ssl.create_default_context(cafile=certifi.where())

pd.set_option('display.max_columns', None)


## 2. Load Dataset

In [ ]:
# Fetch dataset by ID (45 = Heart Disease)
heart_disease = fetch_ucirepo(id=45)

# Split into features (X) and target (y)
X = heart_disease.data.features
y = heart_disease.data.targets

# Combine for convenience
df = pd.concat([X, y], axis=1)

print("Dataset shape:", df.shape)
df.head()

## 3. Basic Information

In [ ]:
df.info()
df.describe()
df.isna().sum()

## 4. Target Variable

In [ ]:
target_col = 'num' 
sns.countplot(x=target_col, data=df)
plt.title('Presence of heart disease (target)')
plt.xlabel('Heart Disease Presence (num)')
plt.show()


## 5. Feature Exploration

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
df[numeric_cols].hist(figsize=(12, 8))
plt.suptitle("Numeric Feature Distributions")
plt.show()

cat_cols = df.select_dtypes(exclude=np.number).columns
for col in cat_cols:
    plt.figure(figsize=(8, 3))
    sns.countplot(x=df[col])
    plt.title(f"Distribution of {col}")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
corr = df.corr(numeric_only=True)['num'].sort_values(ascending=False)
corr

# Preprocessing

## Preprocess Target variable

In [ ]:
def target_bin(x):
    if x>0:
        return 1
    else:
        return 0

df["num"]= df["num"].apply(target_bin) # we binarize the target variable (1= presenze of a heart diseas, 0= not present)
df["diagnosis"]= df["num"] # rename target variable to "diagnosis"
df= df.drop(["num"], axis=1)
df.head()
target_col = 'diagnosis' 
sns.countplot(x=target_col, data=df)
plt.title('Presence of heart disease (target)')
plt.xlabel('Heart Disease Presence (diagnosis)')
plt.show()


### Preprocess Input variables

#### Check for outliers

In [ ]:
plt.figure()
plt.boxplot(df["age"])
plt.title("age") # no outliers

plt.figure()
plt.boxplot(df["trestbps"]) # expected range [90-200]
plt.title("trestbps") # no outliers

plt.figure()
plt.boxplot(df["chol"]) # expect range [100- max 400]
plt.title("chol") # here we can see some outliers which have to be handeled

plt.figure()
plt.boxplot(df["thalach"]) # expected range [80-200]
plt.title("thalach") # no clear outliers here

plt.figure()
plt.boxplot(df["oldpeak"]) # expected range [0-5]
plt.title("oldpeak") # we can see here some upper outliers which have to be handeld

plt.figure()
plt.boxplot(df["ca"].dropna()) # expected range [0-3]
plt.title("ca") # no outliers


In [ ]:
def handle_chol(x): #Cap values a 500 since more extreme values are very unlikly and would highly influence our model
    if x>500:
        return 500
    else:
        return x

def handle_oldpeak(x): #Cap values a 5.5 since more extreme values are very unlikly and would highly influence our model
    if x>5.5:
        return 5.5
    else:
        return x

df["chol"]= df["chol"].apply(handle_chol)
df["oldpeak"]= df["oldpeak"].apply(handle_oldpeak)

# check again visually for outliers
plt.figure()
plt.boxplot(df["chol"]) # expect range [100- max 400]
plt.title("chol") # here we can see some outliers which have to be handeled


plt.figure()
plt.boxplot(df["oldpeak"]) # expected range [0-5]
plt.title("oldpeak") # we can see here some upper outliers which have to be handeld

### Handle missing values

In [ ]:
contains_value = (df == '?').any().any()
contains_value #check if ther are any ? which could be potential null values

In [ ]:
pd.isna(df).any()


In [ ]:
df[pd.isna(df).any(axis=1)]

Here we can see that there are some missing values in the ca and thal columns. By inspecting the dataset we can see no logical meaningful reason for the NA values.

In [ ]:
print("Number of NA values in ca:")
print(df["ca"].isna().sum())
print("Number of NA values in thal:")
print(df["thal"].isna().sum())

We see that there are 4 missing values in the ca column and 2 missing values in the thal column. That are quite few, thats why we decided to just drop the rows with missing values.

In [ ]:
df=df.dropna(subset=["ca","thal"])
pd.isna(df).any()


### Split data into train and test sets

In [ ]:
X = df.drop("diagnosis", axis=1)  # features
y = df["diagnosis"]               # diagnosis

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,       # 20% of data for testing
    random_state=42,     # ensures reproducibility
    stratify=y           # keeps same class proportions in train/test
)

y_train.head()

### Making a scaled dataset

In [ ]:
X_train_scaled=X_train.copy()
X_test_scaled= X_test.copy()
scaler = StandardScaler()
numeric_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak'] # scale all numerical values exept ca
X_train_scaled[numeric_cols]= scaler.fit_transform(X_train_scaled[numeric_cols])
X_test_scaled[numeric_cols]= scaler.fit_transform(X_test_scaled[numeric_cols])
X_test_scaled.describe()

We now have scaled X in two new datasets all numerical values with a standard scaler such that mean is 0 and standard deviation is 1. WE have not included ca in the scalling since it only takes the values 0,1,2,3 and thus it would make a real difference to scale.

# Model Training and Evaluation

We train three classifiers — Decision Tree, KNN, and Naive Bayes — and check how well they predict loan grades.

## Decision Tree Classifier

In [ ]:

# Hyperparameter lists
max_depths = [2,3,5,10, None]
min_splits = [2,3,5,10]
min_leafs = [3,5, 10]
criterions = ["gini", "entropy"]
weights = [None, "balanced"]

results = []

# Loop through all combinations
for depth, split, leaf, crit, weight in itertools.product(max_depths, min_splits, min_leafs, criterions, weights):
    
    clf = DecisionTreeClassifier(
        max_depth=depth,
        min_samples_split=split,
        min_samples_leaf=leaf,
        criterion=crit,
        class_weight=weight,
        random_state=42
    )
    clf.fit(X_train, y_train)
    
    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:,1]  # probability for AUC
    
    results.append({
        "max_depth": depth,
        "min_samples_split": split,
        "min_samples_leaf": leaf,
        "criterion": crit,
        "class_weight": str(weight),
        "accuracy": accuracy_score(y_test, y_pred),
        "macro_f1": f1_score(y_test, y_pred, average='macro'),
        "weighted_f1": f1_score(y_test, y_pred, average='weighted'),
        "auc": roc_auc_score(y_test, y_prob)
    })

# Convert to DataFrame
results_df = pd.DataFrame(results)
res_df_ordered= results_df.copy()
res_df_ordered.sort_values(by="macro_f1", ascending=False, inplace=True)
res_df_ordered = res_df_ordered.set_index(["class_weight","criterion","max_depth","min_samples_split","min_samples_leaf"])
res_df_ordered= res_df_ordered.sort_index()
res_df_ordered.head(50)

In [ ]:
# Plots for visualization

## Headmap for the differnt min/max values:
leaf_values = sorted(results_df["min_samples_leaf"].unique())

# Create a single figure with 1 row and N columns (one subplot per leaf value)
fig, axes = plt.subplots(1, len(leaf_values), figsize=(6 * len(leaf_values), 4), sharey=True)

for ax, leaf in zip(axes, leaf_values):
    subset = results_df[results_df["min_samples_leaf"] == leaf]
    heatmap_data = subset.pivot_table(
        values="macro_f1",
        index="max_depth",
        columns="min_samples_split"
    )
    
    sns.heatmap(heatmap_data, annot=True, fmt=".3f", cmap="YlGnBu", ax=ax)
    ax.set_title(f"min_samples_leaf = {leaf}")
    ax.set_xlabel("min_samples_split")
    ax.set_ylabel("max_depth")

plt.suptitle("Macro F1 by max_depth and min_samples_split for different min_samples_leaf values", y=1.05)
plt.tight_layout()
plt.show()


## Bar chart to show the effect of differnt criterions
sns.barplot(x='class_weight', y='macro_f1', hue='criterion', data=results_df)
plt.title("Effect of class_weight and criterion on Macro F1")
plt.show()

## Plot to show the influnce of weighting the classes
sns.scatterplot(x='weighted_f1', y='macro_f1', hue='class_weight', style='criterion', data=results_df)
plt.title("Weighted F1 vs Macro F1 across Decision Tree variants (class_weight)")
plt.show()



We numerically search the best model dependent on the various evaluation metrics (macro_f1, weighted_f1, accuracy, auc).:

In [ ]:
# Choose the best combination of hyper paramters model:
hyperparams = ["max_depth","min_samples_split","min_samples_leaf","criterion","class_weight"]
# Pick the row with the highest Macro F1
best_model = results_df.loc[results_df['macro_f1'].idxmax()]

print("Optimal hyperparameter combination based on Macro F1:")
print(best_model[hyperparams])

# Pick the row with the highest weighted_f1
best_model = results_df.loc[results_df['weighted_f1'].idxmax()]

print("Optimal hyperparameter combination based on weighted_f1:")
print(best_model[hyperparams])

# Pick the row with the highest accuracy
best_model = results_df.loc[results_df['accuracy'].idxmax()]

print("Optimal hyperparameter combination based on accuracy:")
print(best_model[hyperparams])

# Pick the row with the highest auc
best_model = results_df.loc[results_df['auc'].idxmax()]

print("Optimal hyperparameter combination based on auc:")
print(best_model[hyperparams])


We fit the choosen best model and look what the most importnat features are:

In [ ]:
# Fit the evaluated optical model and get the most important attributes
clf_best = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=2,
    min_samples_leaf=5,
    criterion="gini",
    class_weight=None,
    random_state=42
)

clf_best.fit(X_train, y_train)

# Get feature importances
importances = clf_best.feature_importances_

# Combine with feature names
feature_importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print("Top important features:")
print(feature_importance_df.head(10)) 

We compare the evaluation metrics for the unscaled and scaled dataset:

In [ ]:
#Resulting evaulation values for the best selected model
test_pred = clf_best.predict(X_test)

print("Classification report for the UNSCALED data: ")
print(classification_report(y_test, test_pred))


# Apply the best model on the scaled data (shoulnt make any big difference)
clf_best_scaled = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=2,
    min_samples_leaf=5,
    criterion="gini",
    class_weight=None,
    random_state=42
)

clf_best_scaled.fit(X_train_scaled, y_train)

test_pred_scaled = clf_best_scaled.predict(X_test_scaled)

print("Classification report for the SCALED data: ")
print(classification_report(y_test, test_pred_scaled))

In [ ]:

plt.figure(figsize=(20, 10))
plot_tree(
    clf_best,
    filled=True,
    rounded=True,
    class_names=[str(cls) for cls in clf_best.classes_],
    feature_names=X_train.columns,
    fontsize=10
)
plt.title("Decision Tree (Best Model)")
plt.show()